# Approach D — Interactive Case Viewer\n\nCompares MRI · Ground Truth · Prediction for test cases.  \nUse the sliders to scroll slices; use the dropdowns to switch cases and view axis."

In [ ]:
import sys
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display

REPO = Path("..").resolve()   # pancrea_cyst root
PRED_DIR = REPO / "approach_d/predictions"
TEST_TXT = REPO / "data/test.txt"

# ── load test-set map ─────────────────────────────────────────────────────────
gt_map = {}
for line in TEST_TXT.read_text().splitlines()[1:]:
    line = line.strip()
    if not line: continue
    p = line.split(",")
    stem = Path(p[0]).name.replace(".nii.gz", "")
    gt_map[stem] = (Path(p[0]), Path(p[1]))

print(f"Test cases loaded: {len(gt_map)}")

# ── helpers ───────────────────────────────────────────────────────────────────
def load(path):
    return np.asarray(nib.load(str(path)).dataobj).astype(np.float32)

def norm(v):
    lo, hi = np.percentile(v, 1), np.percentile(v, 99)
    return np.clip((v - lo) / max(hi - lo, 1e-6), 0, 1)

def get_slice(vol, axis, idx):
    s = int(np.clip(idx, 0, vol.shape[axis] - 1))
    if axis == 0: return np.rot90(vol[s])
    if axis == 1: return np.rot90(vol[:, s])
    return np.rot90(vol[:, :, s])

def overlay(gray, mask, color, alpha=0.45):
    rgb = np.stack([gray]*3, axis=-1)
    if mask.any():
        rgb[mask > 0] = (1-alpha)*rgb[mask > 0] + alpha*np.array(color)
    return rgb

def dice(pred, gt):
    p, g = pred > 0, gt > 0
    denom = p.sum() + g.sum()
    return float(2*(p&g).sum()/denom) if denom > 0 else 1.0

_cache = {}
def get_data(name):
    if name not in _cache:
        img_p, gt_p = gt_map[name]
        pred_p = PRED_DIR / f"{name}.nii.gz"
        img  = norm(load(img_p))
        gt   = (load(gt_p) > 0).astype(np.uint8)
        pred = (load(pred_p) > 0).astype(np.uint8) if pred_p.exists() else np.zeros_like(gt)
        _cache[name] = (img, gt, pred)
    return _cache[name]

print("Helpers ready.")

In [ ]:
%matplotlib inline

AXIS_MAP = {"Axial (Z)": 0, "Coronal (Y)": 1, "Sagittal (X)": 2}
HIGHLIGHT = ["NU28", "NU188", "EMC077", "NU71", "NYU0165"]   # default cases
DEFAULT_CASES = [c for c in HIGHLIGHT if c in gt_map]

case_dd  = widgets.Dropdown(options=DEFAULT_CASES, value=DEFAULT_CASES[0],
                            description="Case:", layout=widgets.Layout(width="180px"))
axis_dd  = widgets.Dropdown(options=list(AXIS_MAP.keys()), value="Axial (Z)",
                            description="View:", layout=widgets.Layout(width="190px"))
slice_sl = widgets.IntSlider(min=0, max=1, value=0, description="Slice",
                             continuous_update=True, layout=widgets.Layout(width="500px"))
out      = widgets.Output()

def best_slice(gt, axis):
    sums = [get_slice(gt, axis, i).sum() for i in range(gt.shape[axis])]
    return int(np.argmax(sums)) if max(sums) > 0 else gt.shape[axis] // 2

def update_slider(name, axis):
    img, gt, pred = get_data(name)
    n = img.shape[axis]
    slice_sl.max   = n - 1
    slice_sl.value = best_slice(gt, axis)

def render(name, axis_label, sl):
    axis = AXIS_MAP[axis_label]
    img, gt, pred = get_data(name)
    sl = int(np.clip(sl, 0, img.shape[axis] - 1))

    dc    = dice(pred, gt)
    n_gt  = int(gt.sum())
    n_pred = int(pred.sum())
    missed = " ⚠ model predicted nothing!" if n_pred == 0 and n_gt > 0 else ""

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("#111")

    titles = ["MRI", "GT  (green)", "Prediction  (orange)"]
    imgs   = [
        np.stack([get_slice(img, axis, sl)]*3, axis=-1),
        overlay(get_slice(img, axis, sl), get_slice(gt,   axis, sl), [0.0, 1.0, 0.0]),
        overlay(get_slice(img, axis, sl), get_slice(pred, axis, sl), [1.0, 0.35, 0.0]),
    ]

    for ax, im, title in zip(axes, imgs, titles):
        ax.imshow(im, vmin=0, vmax=1, interpolation="bilinear")
        ax.set_title(title, color="white", fontsize=11)
        ax.axis("off")
        ax.set_facecolor("black")

    fig.suptitle(
        f"{name}  |  {axis_label}  slice {sl}  |  "
        f"Dice: {dc:.4f}  |  GT: {n_gt} vox  |  Pred: {n_pred} vox{missed}",
        color="white", fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()

def on_change(_):
    name       = case_dd.value
    axis_label = axis_dd.value
    axis       = AXIS_MAP[axis_label]
    update_slider(name, axis)
    with out:
        out.clear_output(wait=True)
        render(name, axis_label, slice_sl.value)

def on_slice(change):
    with out:
        out.clear_output(wait=True)
        render(case_dd.value, axis_dd.value, change["new"])

case_dd.observe(on_change,  names="value")
axis_dd.observe(on_change,  names="value")
slice_sl.observe(on_slice,  names="value")

# initial render
update_slider(case_dd.value, AXIS_MAP[axis_dd.value])
with out:
    render(case_dd.value, axis_dd.value, slice_sl.value)

display(widgets.HBox([case_dd, axis_dd]), slice_sl, out)

## Browse any test case\nChange the `CASES` list below to any subset of the 74 test cases."

In [ ]:
# Edit this list to browse any test cases
CASES = list(gt_map.keys())   # all 74 — or replace with e.g. ["NU28", "AHN54"]

case_dd2  = widgets.Dropdown(options=CASES, value=CASES[0],
                             description="Case:", layout=widgets.Layout(width="180px"))
axis_dd2  = widgets.Dropdown(options=list(AXIS_MAP.keys()), value="Axial (Z)",
                             description="View:", layout=widgets.Layout(width="190px"))
slice_sl2 = widgets.IntSlider(min=0, max=1, value=0, description="Slice",
                              continuous_update=True, layout=widgets.Layout(width="500px"))
out2      = widgets.Output()

def on_change2(_):
    name  = case_dd2.value
    axis  = AXIS_MAP[axis_dd2.value]
    update_slider_w(name, axis, slice_sl2)
    with out2:
        out2.clear_output(wait=True)
        render(name, axis_dd2.value, slice_sl2.value)

def on_slice2(change):
    with out2:
        out2.clear_output(wait=True)
        render(case_dd2.value, axis_dd2.value, change["new"])

def update_slider_w(name, axis, sl_widget):
    img, gt, pred = get_data(name)
    n = img.shape[axis]
    sl_widget.max   = n - 1
    sl_widget.value = best_slice(gt, axis)

case_dd2.observe(on_change2,  names="value")
axis_dd2.observe(on_change2,  names="value")
slice_sl2.observe(on_slice2,  names="value")

update_slider_w(case_dd2.value, AXIS_MAP[axis_dd2.value], slice_sl2)
with out2:
    render(case_dd2.value, axis_dd2.value, slice_sl2.value)

display(widgets.HBox([case_dd2, axis_dd2]), slice_sl2, out2)